# Colab launcher

Thin launcher: clone the repo, install, then run `train.py` through the plan in `run_plan.py`. All logic lives in the repo, so a `git pull` (cell 1) is the only update this notebook ever needs.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Checkpoints and results go to Google Drive. After a disconnect, run cells 1 and 2 again, then the plan cell: finished runs are skipped and an interrupted run resumes from `last.ckpt`.

The plan cell runs everything, in order. The cell after it is the final test-set run, to be used once at the end.

In [ ]:
import os
import torch
from google.colab import drive

# Fail fast: on a CPU runtime training is far slower and the session is wasted.
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."

drive.mount("/content/drive")

REPO_URL = "https://github.com/ori112/representation-learning-course"
BRANCH = "midterm"
REPO_DIR = "/content/representation-learning-course"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}/midterm
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
# Keep outputs on Drive so a disconnect does not lose checkpoints or results.
# The Tiny-ImageNet cache (dataset/) stays local: rebuilding it takes a few minutes.
DRIVE_DIR = "/content/drive/MyDrive/rl_midterm"
for folder in ("outputs", "results"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)
    if not os.path.islink(folder):
        os.symlink(f"{DRIVE_DIR}/{folder}", folder)

In [ ]:
# Run the whole Part A plan from zero, in order (the plan lives in run_plan.py, so this cell never needs editing):
# search -> seed check -> final models -> every evaluation. The test set is NOT included, see the next cell.
# Finished runs are skipped, so after a disconnect run cells 1 and 2 again, then this one.
!python run_plan.py

In [ ]:
# The final numbers on the test split. Run this ONCE, only after the cell above has finished and its results look right.
!python run_plan.py --only test

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/tb_logs